In [2]:
import pymupdf
import pandas as pd
import csv
import json
import re
from pathlib import Path

In [3]:
pdf_path = Path("Flipkart Mobile - 2.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        "PDF file not found. Keep it in the notebook folder."
    )

pdf = pymupdf.open(pdf_path)

print("Total pages:", len(pdf))

Total pages: 15


In [5]:
page_texts = []

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text("text")
    page_texts.append(text)

    print(f"Page {page_number}: {len(text)} characters")

all_text = "\n".join(page_texts)

print("\nTotal extracted characters:", len(all_text))

Page 1: 2400 characters
Page 2: 2385 characters
Page 3: 2728 characters
Page 4: 2709 characters
Page 5: 2326 characters
Page 6: 2482 characters
Page 7: 2354 characters
Page 8: 2519 characters
Page 9: 2451 characters
Page 10: 2466 characters
Page 11: 2521 characters
Page 12: 2461 characters
Page 13: 2469 characters
Page 14: 2521 characters
Page 15: 1002 characters

Total extracted characters: 35808


In [6]:
lines = [
    line.strip()
    for line in all_text.splitlines()
    if line.strip()
]

print("Total non-empty lines:", len(lines))

for i, line in enumerate(lines[:50], start=1):
    print(f"{i}: {line}")

Total non-empty lines: 831
1: brand,model,base_color,processor,screen_size,ROM,RAM,display_size,num_rea
2: r_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_p
3: rice,discount_percent,sales
4: Apple,iPhone SE,Black,Water,Very
5: Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
6: Apple,iPhone 12
7: Mini,Red,Ceramic,Small,64,4,5.4,2,1,2815,4.5,244,57149,0.04,1.39
8: Apple,iPhone SE,Red,Water,Very
9: Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
10: Apple,iPhone
11: XR,Others,iOS,Medium,64,3,6.1,1,1,2942,4.6,5366,42999,0.1,23.07
12: Apple,iPhone
13: 12,Red,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
14: Apple,iPhone
15: 12,Blue,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
16: Apple,iPhone
17: 12,White,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
18: Apple,iPhone
19: 12,Green,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
20: Apple,iPhone
21: 12,Blue,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
22

In [7]:
EXPECTED_COLUMNS = 16
EXPECTED_COMMAS = EXPECTED_COLUMNS - 1

candidate_rows = []
buffer = ""

for line in lines:
    buffer = f"{buffer} {line}".strip() if buffer else line

    if buffer.count(",") >= EXPECTED_COMMAS:
        candidate_rows.append(buffer)
        buffer = ""

if buffer:
    print("Incomplete leftover text:")
    print(buffer)

print("Candidate records:", len(candidate_rows))

for row in candidate_rows[:10]:
    print(row)

Candidate records: 431
brand,model,base_color,processor,screen_size,ROM,RAM,display_size,num_rea r_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_p rice,discount_percent,sales
Apple,iPhone SE,Black,Water,Very Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
Apple,iPhone 12 Mini,Red,Ceramic,Small,64,4,5.4,2,1,2815,4.5,244,57149,0.04,1.39
Apple,iPhone SE,Red,Water,Very Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
Apple,iPhone XR,Others,iOS,Medium,64,3,6.1,1,1,2942,4.6,5366,42999,0.1,23.07
Apple,iPhone 12,Red,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
Apple,iPhone 12,Blue,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
Apple,iPhone 12,White,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
Apple,iPhone 12,Green,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
Apple,iPhone 12,Blue,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15


In [8]:
import pandas as pd

# Correct the broken header names
columns = [
    col.replace(" ", "").strip()
    for col in candidate_rows[0].split(",")
]

# Build a DataFrame only from rows with exactly 16 fields
valid_rows = []
invalid_rows = []

for i, row in enumerate(candidate_rows[1:], start=2):
    values = [value.strip() for value in row.split(",")]

    if len(values) == EXPECTED_COLUMNS:
        valid_rows.append(values)
    else:
        invalid_rows.append((i, len(values), row))

df = pd.DataFrame(valid_rows, columns=columns)

print("Data rows:", len(df))
print("Invalid rows:", len(invalid_rows))
print("Columns:", df.columns.tolist())

print("\nInvalid row examples:")
for item in invalid_rows[:5]:
    print(item)

Data rows: 430
Invalid rows: 0
Columns: ['brand', 'model', 'base_color', 'processor', 'screen_size', 'ROM', 'RAM', 'display_size', 'num_rear_camera', 'num_front_camera', 'battery_capacity', 'ratings', 'num_of_ratings', 'sales_price', 'discount_percent', 'sales']

Invalid row examples:


In [9]:
parsed_rows = []
bad_rows = []

for row in candidate_rows:
    try:
        fields = next(csv.reader([row]))

        if len(fields) == EXPECTED_COLUMNS:
            parsed_rows.append(
                [value.strip() for value in fields]
            )
        else:
            bad_rows.append({
                "field_count": len(fields),
                "row": row
            })

    except csv.Error as error:
        bad_rows.append({
            "field_count": "parse error",
            "row": row
        })

print("Valid rows including header:", len(parsed_rows))
print("Invalid rows:", len(bad_rows))

for item in bad_rows[:10]:
    print(item)

Valid rows including header: 431
Invalid rows: 0


In [10]:
if not parsed_rows:
    raise ValueError("No valid records found.")

columns = [
    re.sub(r"\s+", "", name.strip())
    for name in parsed_rows[0]
]

expected_columns = [
    "brand",
    "model",
    "base_color",
    "processor",
    "screen_size",
    "ROM",
    "RAM",
    "display_size",
    "num_rear_camera",
    "num_front_camera",
    "battery_capacity",
    "ratings",
    "num_of_ratings",
    "sales_price",
    "discount_percent",
    "sales"
]

print("Detected columns:", columns)

if columns != expected_columns:
    raise ValueError(
        "Header mismatch. Inspect the extracted header before proceeding."
    )

data_rows = parsed_rows[1:]

df = pd.DataFrame(data_rows, columns=columns)

print("Dataset shape:", df.shape)
display(df.head(10))

Detected columns: ['brand', 'model', 'base_color', 'processor', 'screen_size', 'ROM', 'RAM', 'display_size', 'num_rear_camera', 'num_front_camera', 'battery_capacity', 'ratings', 'num_of_ratings', 'sales_price', 'discount_percent', 'sales']
Dataset shape: (430, 16)


,brand,model,base_color,processor,screen_size,ROM,RAM,display_size,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
0,Apple,iPhone SE,Black,Water,Very Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
1,Apple,iPhone 12 Mini,Red,Ceramic,Small,64,4,5.4,2,1,2815,4.5,244,57149,0.04,1.39
2,Apple,iPhone SE,Red,Water,Very Small,64,2,4.7,1,1,1800,4.5,38645,32999,0.17,127.52
3,Apple,iPhone XR,Others,iOS,Medium,64,3,6.1,1,1,2942,4.6,5366,42999,0.1,23.07
4,Apple,iPhone 12,Red,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
5,Apple,iPhone 12,Blue,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
6,Apple,iPhone 12,White,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
7,Apple,iPhone 12,Green,Ceramic,Medium,64,4,6.1,2,1,2815,4.6,745,64149,0.02,4.78
8,Apple,iPhone 12,Blue,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15
9,Apple,iPhone 12,Black,Ceramic,Medium,128,4,6.1,2,1,2815,4.6,745,69149,0.02,5.15


In [11]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nBlank values:")
display(
    (df.astype(str).apply(lambda col: col.str.strip() == ""))
    .sum()
    .to_frame("blank_values")
)

print("\nData types before conversion:")
print(df.dtypes)

Missing values:


,missing_values
brand,0
model,0
base_color,0
processor,0
screen_size,0
ROM,0
RAM,0
display_size,0
num_rear_camera,0
num_front_camera,0



Blank values:


,blank_values
brand,0
model,0
base_color,0
processor,0
screen_size,0
ROM,0
RAM,0
display_size,0
num_rear_camera,0
num_front_camera,0



Data types before conversion:
brand               str
model               str
base_color          str
processor           str
screen_size         str
ROM                 str
RAM                 str
display_size        str
num_rear_camera     str
num_front_camera    str
battery_capacity    str
ratings             str
num_of_ratings      str
sales_price         str
discount_percent    str
sales               str
dtype: object


In [12]:
display(df.sample(min(10, len(df)), random_state=42))

,brand,model,base_color,processor,screen_size,ROM,RAM,display_size,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
422,Xiaomi,Redmi Note 9 Pro,Blue,Qualcomm,Large,128,6,6.7,4,1,5020,4.3,434,14999,0.06,0.65
75,Poco,X3 Pro,Blue,Qualcomm,Large,128,8,6.7,4,1,5160,4.4,4780,20999,0.19,10.04
177,Realme,GT 5G,Silver,Qualcomm,Medium,128,8,6.4,3,1,4500,4.3,50,37999,0.07,0.19
30,Apple,iPhone 12 Mini,Blue,Ceramic,Small,64,4,5.4,2,1,2815,4.5,244,57149,0.04,1.39
361,Samsung,Galaxy A71,Blue,Qualcomm,Large,128,8,6.7,4,1,4500,4.4,3510,27499,0.21,9.65
380,Xiaomi,Redmi Note 9 Pro,White,Qualcomm,Large,64,4,6.7,4,1,5020,4.4,6106,16487,0.03,10.07
155,Realme,Narzo 30 Pro 5G,Silver,MediaTek,Large,64,6,6.5,3,1,5000,4.3,12805,17149,0.09,21.96
152,.91 Realme,GT Master Edition,White,Qualcomm,Medium,256,8,6.4,3,1,4300,4.3,105,29999,0.06,0.31
165,Realme,Narzo 30 Pro 5G,Black,MediaTek,Large,128,8,6.5,3,1,5000,4.3,5485,20149,0.08,11.05
175,Realme,X7 Max,Others,MediaTek,Medium,128,8,6.4,3,1,4500,4.3,3684,27149,0.09,10.0


In [13]:
chunks = []

for index, row in df.iterrows():

    text = (
        f"Brand: {row['brand']}. "
        f"Model: {row['model']}. "
        f"Color: {row['base_color']}. "
        f"Processor: {row['processor']}. "
        f"Screen size category: {row['screen_size']}. "
        f"ROM: {row['ROM']} GB. "
        f"RAM: {row['RAM']} GB. "
        f"Display size: {row['display_size']} inches. "
        f"Rear cameras: {row['num_rear_camera']}. "
        f"Front cameras: {row['num_front_camera']}. "
        f"Battery: {row['battery_capacity']} mAh. "
        f"Rating: {row['ratings']}. "
        f"Number of ratings: {row['num_of_ratings']}. "
        f"Sales price: ₹{row['sales_price']}. "
        f"Discount percent: {row['discount_percent']}. "
        f"Sales: {row['sales']}."
    )
    

    chunks.append({
        "page_content": text,
        "metadata": {
            "row_id": int(index),
            "source": pdf_path.name,
            "brand": str(row["brand"]),
            "model": str(row["model"]),
            "RAM": None if pd.isna(row["RAM"]) else int(row["RAM"]),
            "ROM": None if pd.isna(row["ROM"]) else int(row["ROM"]),
            "battery_capacity": (
                None if pd.isna(row["battery_capacity"])
                else int(row["battery_capacity"])
            ),
            "sales_price": (
                None if pd.isna(row["sales_price"])
                else float(row["sales_price"])
            )
        }
    })

print("Documents created:", len(chunks))
print(chunks[0])

Documents created: 430
{'page_content': 'Brand: Apple. Model: iPhone SE. Color: Black. Processor: Water. Screen size category: Very Small. ROM: 64 GB. RAM: 2 GB. Display size: 4.7 inches. Rear cameras: 1. Front cameras: 1. Battery: 1800 mAh. Rating: 4.5. Number of ratings: 38645. Sales price: ₹32999. Discount percent: 0.17. Sales: 127.52.', 'metadata': {'row_id': 0, 'source': 'Flipkart Mobile - 2.pdf', 'brand': 'Apple', 'model': 'iPhone SE', 'RAM': 2, 'ROM': 64, 'battery_capacity': 1800, 'sales_price': 32999.0}}


In [ ]:
df.to_csv(
    "flipkart_mobiles_cleaned.csv",
    index=False,
    encoding="utf-8"
)

with open(
    "flipkart_mobiles_chunks.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        chunks,
        file,
        ensure_ascii=False,
        indent=2
    )

print("CSV saved: flipkart_mobiles_cleaned.csv")
print("JSON saved: flipkart_mobiles_chunks.json")
print("Total records:", len(df))